In [ ]:
# ----------------------------------------------------------------------------
# 1. Installing required packages
# ----------------------------------------------------------------------------

#!pip install torch torchvision torchaudio --index-rl https://download.pytorch.org/whl/cu118
#!pip install torch-scatter torch-sparse torch-cluster torch-spline-conv -f https://data.pyg.org/whl/torch-2.0.0+cu118.html
!pip install torch-geometric networkx pandas numpy scikit-learn tqdm codecarbon openpyxl


In [ ]:
# ============================================================================
# CELL 2: Imports and common definitions (shared by all experiments)
# ============================================================================
# -*- coding: utf-8 -*-
import numpy as np
import pandas as pd
import os
import time
import uuid
import copy
import random
import sys

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.utils import from_networkx
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GATConv, SAGEConv, GCNConv, global_mean_pool
import networkx as nx
from torch_geometric.loader import DataLoader as PyGDataLoader
from torch.utils.data import DataLoader
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             precision_score, recall_score, f1_score,
                             roc_auc_score, matthews_corrcoef)
from tqdm import tqdm
from torch_geometric.nn import GCNConv, global_mean_pool, global_add_pool

# CodeCarbon is optional
try:
    from codecarbon import EmissionsTracker
    CODECARBON_AVAILABLE = True
except ImportError:
    CODECARBON_AVAILABLE = False
    print("codecarbon not installed. Energy tracking will be dummy.")


DATA_DIR = "/content/data/"          # folder containing all .txt files
OUTPUT_DIR = "/content/results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CONTROLS_DIR = DATA_DIR
PATIENTS_DIR = DATA_DIR

# === IDs ===
coma_ids = [11, 13, 15, 17, 19, 21, 25, 26, 28, 35, 36, 4, 6, 7, 8]
mcs_ids = [18, 22, 24, 2, 30, 31, 3, 5, 9]
all_ill= [11, 13, 15, 17, 19, 21, 25, 26, 28, 35, 36, 4, 6, 7, 8,18, 22, 24, 2, 30, 31, 3, 5, 9]
num_nodes = 105
NUM_VIEWS = 4

# ---------------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------------
def reconstruct_symmetric_matrix(upper_tri, size=105):
    matrix = np.zeros((size, size))
    idx = np.triu_indices(size, k=1)
    matrix[idx] = upper_tri
    matrix = matrix + matrix.T
    np.fill_diagonal(matrix, 1.0)
    return matrix

def extract_nums(s):
    parts = s.split('R')[1:]
    return tuple(map(int, parts))

def get_patient_id(index_name):
    return int(index_name.replace('patient', '').replace('_exam1', ''))

def create_graph_from_matrix(matrix):
    rows, cols = matrix.shape
    G = nx.Graph()
    G.add_nodes_from(range(rows))
    for i in range(rows):
        for j in range(i + 1, cols):
            if matrix[i, j] != 0:
                G.add_edge(i, j, weight=float(abs(matrix[i, j])))
    for node in G.nodes:
        connection_profile = matrix[node].astype(float).tolist()
        G.nodes[node]['x'] = connection_profile
    return G

def measure_energy(func, *args, **kwargs):
    if not CODECARBON_AVAILABLE:
        t0 = time.perf_counter()
        result = func(*args, **kwargs)
        runtime = time.perf_counter() - t0
        metrics = {
            "runtime_sec": runtime,
            "emissions_kg": float("nan"),
            "cpu_energy_kwh": float("nan"),
            "gpu_energy_kwh": float("nan"),
            "ram_energy_kwh": float("nan"),
        }
        return result, metrics
    uid = str(uuid.uuid4())[:8]
    csv_path = f"emissions_{uid}.csv"
    tracker = EmissionsTracker(
        measure_power_secs=1,
        tracking_mode="process",
        save_to_file=True,
        output_file=csv_path,
        log_level="error"
    )
    tracker.start()
    t0 = time.perf_counter()
    result = func(*args, **kwargs)
    emissions_kg = tracker.stop()
    runtime = time.perf_counter() - t0
    df = pd.read_csv(csv_path)
    last = df.iloc[-1]
    metrics = {
        "runtime_sec": runtime,
        "emissions_kg": emissions_kg,
        "cpu_energy_kwh": last.get("cpu_energy", float("nan")),
        "gpu_energy_kwh": last.get("gpu_energy", float("nan")),
        "ram_energy_kwh": last.get("ram_energy", float("nan")),
    }
    try:
        os.remove(csv_path)
    except Exception:
        pass
    return result, metrics

# ---------------------------------------------------------------------------
# Dataset and collate
# ---------------------------------------------------------------------------
class MultiQuantileDataset(torch.utils.data.Dataset):
    def __init__(self, subject_graphs, labels):
        self.subject_graphs = subject_graphs
        self.labels = labels
    def __len__(self):
        return len(self.subject_graphs)
    def __getitem__(self, idx):
        return self.subject_graphs[idx], self.labels[idx]

def collate_multi_quantile(batch):
    subj_list, labels = zip(*batch)
    num_q = len(subj_list[0])
    batched = [Batch.from_data_list([subj[q] for subj in subj_list]) for q in range(num_q)]
    y = torch.tensor(labels, dtype=torch.long)
    return batched, y

# ---------------------------------------------------------------------------
# Base GCN (used by all models)
# ---------------------------------------------------------------------------
class GCN(nn.Module):
    def __init__(self, in_channels, hidden_channels=105, num_layers=3):
        super().__init__()
        layers = []
        layers.append(GCNConv(in_channels, hidden_channels))
        for _ in range(num_layers - 2):
            layers.append(GCNConv(hidden_channels, hidden_channels))
        layers.append(GCNConv(hidden_channels, hidden_channels))
        self.layers = nn.ModuleList(layers)
        self.classifier = nn.Linear(hidden_channels, 2)

    def forward(self, data, return_embedding=False):
        x, edge_index, edge_weight, batch = (
            data.x, data.edge_index, data.edge_weight, data.batch)
        for i, conv in enumerate(self.layers):
            x = conv(x, edge_index, edge_weight)
            if i < len(self.layers) - 1:
                x = F.relu(x)
        emb = global_mean_pool(x, batch)
        if return_embedding:
            return self.classifier(emb), emb
        return self.classifier(emb)

# ---------------------------------------------------------------------------
# Fusion modules
# ---------------------------------------------------------------------------
class QuantileAttention(nn.Module):
    def __init__(self, embedding_dim, attention_dim):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(embedding_dim, attention_dim),
            nn.Tanh(),
            nn.Linear(attention_dim, 1)
        )
    def forward(self, embeddings):
        scores = self.attention(embeddings)
        attention_weights = torch.softmax(scores, dim=1)
        fused_embedding = torch.sum(attention_weights * embeddings, dim=1)
        return fused_embedding, attention_weights

class SumFusion(nn.Module):
    def __init__(self, embedding_dim, attention_dim=None):
        super().__init__()
    def forward(self, embeddings):
        fused_embedding = embeddings.sum(dim=1)
        attention_weights = None
        return fused_embedding, attention_weights

class ConcatFusion(nn.Module):
    def __init__(self, embedding_dim, attention_dim=None):
        super().__init__()
        self.proj = nn.Linear(embedding_dim * NUM_VIEWS, embedding_dim)
    def forward(self, embeddings):
        b, q, d = embeddings.shape
        flat = embeddings.reshape(b, q * d)
        fused_embedding = self.proj(flat)
        attention_weights = None
        return fused_embedding, attention_weights

# ---------------------------------------------------------------------------
# Model variants: Parallel (separate GCN per view) and Shared (one GCN for all)
# ---------------------------------------------------------------------------
class ParallelMultiQuantileGCN(nn.Module):
    def __init__(self, in_channels, hidden_channels=105, num_layers=3,
                 embedding_dim=105, attention_dim=32, num_classes=2, fusion='attention'):
        super().__init__()
        self.gcns = nn.ModuleList([
            GCN(in_channels, hidden_channels, num_layers) for _ in range(NUM_VIEWS)
        ])
        if fusion == 'attention':
            self.fusion = QuantileAttention(embedding_dim, attention_dim)
        elif fusion == 'sum':
            self.fusion = SumFusion(embedding_dim)
        elif fusion == 'concat':
            self.fusion = ConcatFusion(embedding_dim)
        else:
            raise ValueError(f"Unknown fusion: {fusion}")
        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, num_classes)
        )

    def forward(self, quantile_graphs):
        embeddings = []
        for gcn, graph in zip(self.gcns, quantile_graphs):
            _, emb = gcn(graph, return_embedding=True)
            embeddings.append(emb)
        embeddings = torch.stack(embeddings, dim=1)
        fused_embedding, attention_weights = self.fusion(embeddings)
        logits = self.classifier(fused_embedding)
        return logits, attention_weights, embeddings

class SharedMultiQuantileGCN(nn.Module):
    def __init__(self, in_channels, hidden_channels=105, num_layers=3,
                 embedding_dim=105, attention_dim=32, num_classes=2, fusion='attention'):
        super().__init__()
        self.gcn = GCN(in_channels, hidden_channels, num_layers)
        if fusion == 'attention':
            self.fusion = QuantileAttention(embedding_dim, attention_dim)
        elif fusion == 'sum':
            self.fusion = SumFusion(embedding_dim)
        elif fusion == 'concat':
            self.fusion = ConcatFusion(embedding_dim)
        else:
            raise ValueError(f"Unknown fusion: {fusion}")
        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, num_classes)
        )

    def forward(self, quantile_graphs):
        embeddings = []
        for graph in quantile_graphs:
            _, emb = self.gcn(graph, return_embedding=True)
            embeddings.append(emb)
        embeddings = torch.stack(embeddings, dim=1)
        fused_embedding, attention_weights = self.fusion(embeddings)
        logits = self.classifier(fused_embedding)
        return logits, attention_weights, embeddings

# ---------------------------------------------------------------------------
# Training loop
# ---------------------------------------------------------------------------
def train_multi_quantile_gcn(model, train_loader, val_loader, device="cuda", epochs=200, lr=0.001):
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    best_state, best_val_acc = None, 0.0
    for ep in range(epochs):
        model.train()
        for quantile_graphs, yb in train_loader:
            quantile_graphs = [g.to(device) for g in quantile_graphs]
            yb = yb.to(device)
            optimizer.zero_grad()
            logits, _, _ = model(quantile_graphs)
            loss = loss_fn(logits, yb)
            loss.backward()
            optimizer.step()
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for quantile_graphs, yb in val_loader:
                quantile_graphs = [g.to(device) for g in quantile_graphs]
                yb = yb.to(device)
                logits, _, _ = model(quantile_graphs)
                pred = logits.argmax(dim=1)
                correct += (pred == yb).sum().item()
                total += yb.size(0)
        acc = correct / total if total > 0 else 0.0
        if ep % 10 == 0:
            print(f"Epoch {ep}: train loss = {loss.item():.4f}, val_acc = {acc:.4f}")
        if acc > best_val_acc:
            best_val_acc = acc
            best_state = copy.deepcopy(model.state_dict())
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

# ============================================================================
# CELL 3: Load data once (shared by all experiments)
# ============================================================================
# This cell builds the subject_graphs and labels (y) – used by all runs.
print("Loading data...")
all_ill= [11, 13, 15, 17, 19, 21, 25, 26, 28, 35, 36, 4, 6, 7, 8,18, 22, 24, 2, 30, 31, 3, 5, 9]
NUM_QUANTILES = (3,5,9)
per_quantile_matrices = {}
y = None

for q in NUM_QUANTILES:
    file_path1 = os.path.join( f"Coma_controls_inter_corr_quantile0pt{q}_tab.txt")
    file_path2 = os.path.join( f"Coma_patients_inter_corr_quantile0pt{q}_tab_exam1.txt")
    if not os.path.isfile(file_path1):
        raise FileNotFoundError(file_path1)
    if not os.path.isfile(file_path2):
        raise FileNotFoundError(file_path2)
    df1 = pd.read_csv(file_path1, sep='\s+', index_col=0)
    df2 = pd.read_csv(file_path2, sep='\s+', index_col=0)
    sorted_columns = sorted(df1.columns, key=extract_nums)
    df1 = df1[sorted_columns]
    df2 = df2[sorted_columns]
    data2 = df2[[get_patient_id(idx) in all_ill for idx in df2.index]]
    corr_mats = np.empty((44, 105, 105))
    y = np.empty(44)
    for i in range(20):
        corr_mats[i] = reconstruct_symmetric_matrix(df1.iloc[i], size=105)
        y[i] = 0
    for i in range(20, 44):
        corr_mats[i] = reconstruct_symmetric_matrix(data2.iloc[i - 20], size=105)
        y[i] = 1
    per_quantile_matrices[q] = corr_mats

# CA data
file_path1ca = os.path.join( "coma_Controls_corr_CA_tab_all_subjects.txt")
file_path2ca = os.path.join( "coma_patients_corr_CA_tab_all_subjects.txt")
if not os.path.isfile(file_path1ca):
    raise FileNotFoundError(file_path1ca)
if not os.path.isfile(file_path2ca):
    raise FileNotFoundError(file_path2ca)
df1ca = pd.read_csv(file_path1ca, sep='\s+', header=None, index_col=False)
df2ca = pd.read_csv(file_path2ca, sep='\s+', header=None, index_col=False)
df1ca.columns = df1.columns
df2ca.columns = df1.columns
corr_mats = np.empty((44, 105, 105))
y = np.empty(44)
for i in range(20):
    corr_mats[i] = reconstruct_symmetric_matrix(df1ca.iloc[i], size=105)
    y[i] = 0
for i in range(20, 44):
    corr_mats[i] = reconstruct_symmetric_matrix(df2ca.iloc[i - 20], size=105)
    y[i] = 1
per_quantile_matrices[4] = corr_mats

# Build subject graphs
subject_graphs = []
for subj_idx in range(44):
    quantile_list = []
    for q in (3,5,9,4):
        g = create_graph_from_matrix(per_quantile_matrices[q][subj_idx])
        data = from_networkx(g, group_node_attrs=['x'], group_edge_attrs=['weight'])
        data.edge_weight = data.edge_attr.squeeze(-1)
        quantile_list.append(data)
    subject_graphs.append(quantile_list)

print("Data loaded. Number of subjects:", len(subject_graphs))
X_idx = np.arange(len(subject_graphs))
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ============================================================================
# CELL 4: Define the experiment runner function
# ============================================================================
def run_experiment(arch, fusion, suffix):
    """
    arch: 'parallel' or 'shared'
    fusion: 'attention', 'sum', 'concat'
    suffix: string used in output filenames
    """
    print(f"\n{'='*60}")
    print(f"Running: arch={arch}, fusion={fusion} (suffix={suffix})")
    print('='*60)

    # Choose model class
    if arch == 'parallel':
        ModelClass = ParallelMultiQuantileGCN
    elif arch == 'shared':
        ModelClass = SharedMultiQuantileGCN
    else:
        raise ValueError("arch must be 'parallel' or 'shared'")

    outer_seed_global = random.randint(1, 10_000_000)
    print("OUTER SEED =", outer_seed_global)

    cv_outer = RepeatedStratifiedKFold(
        n_splits=5, n_repeats=10, random_state=outer_seed_global
    )

    all_rows = []
    outer_split_number = 1

    for train_val_idx, test_idx in cv_outer.split(X_idx, y):
        repeat_idx = (outer_split_number - 1) // 5 + 1
        fold_idx = (outer_split_number - 1) % 5 + 1
        inner_seed = outer_split_number
        print(f"\n===== REPEAT {repeat_idx} - FOLD {fold_idx} =====")

        inner_cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=inner_seed)
        for train_rel, val_rel in inner_cv.split(np.arange(len(train_val_idx)), y[train_val_idx]):
            break

        train_idx = train_val_idx[train_rel]
        val_idx = train_val_idx[val_rel]

        train_ds = MultiQuantileDataset([subject_graphs[i] for i in train_idx], y[train_idx])
        val_ds = MultiQuantileDataset([subject_graphs[i] for i in val_idx], y[val_idx])
        test_ds = MultiQuantileDataset([subject_graphs[i] for i in test_idx], y[test_idx])

        train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_multi_quantile)
        val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, collate_fn=collate_multi_quantile)
        test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=collate_multi_quantile)

        in_channels = subject_graphs[0][0].x.shape[1]

        def run_train():
            model = ModelClass(
                in_channels=in_channels,
                hidden_channels=105,
                num_layers=3,
                embedding_dim=105,
                attention_dim=32,
                fusion=fusion
            )
            return train_multi_quantile_gcn(model, train_loader, val_loader, device=device)

        (final_model, train_energy) = measure_energy(run_train)

        final_model.eval()
        y_pred, y_true, attn_records = [], [], []
        with torch.no_grad():
            for quantile_graphs, yb in test_loader:
                quantile_graphs = [g.to(device) for g in quantile_graphs]
                yb = yb.to(device)
                logits, attn_weights, _ = final_model(quantile_graphs)
                pred = logits.argmax(dim=1).item()
                y_pred.append(pred)
                y_true.append(yb.item())
                if attn_weights is not None:
                    attn_records.append(attn_weights.squeeze().detach().cpu().numpy())
                else:
                    attn_records.append([float('nan')] * NUM_VIEWS)  # placeholder for sum/concat

        for i, idx in enumerate(test_idx):
            row = {
                "repeat": repeat_idx,
                "fold": fold_idx,
                "outer_seed_global": outer_seed_global,
                "inner_seed": inner_seed,
                "sample_index": int(idx),
                "y_true": int(y_true[i]),
                "y_pred": int(y_pred[i]),
                "train_cpu_energy_kWh": train_energy["cpu_energy_kwh"],
                "train_gpu_energy_kWh": train_energy["gpu_energy_kwh"],
                "train_ram_energy_kWh": train_energy["ram_energy_kwh"],
                "train_runtime_sec": train_energy["runtime_sec"],
                "train_emissions_kg": train_energy["emissions_kg"],
            }
            # Attention weights (if available) or NaNs
            for qi, w in enumerate(attn_records[i], start=1):
                row[f"attn_q{qi}"] = float(w)
            all_rows.append(row)

        outer_split_number += 1

    df = pd.DataFrame(all_rows)
    out_file = os.path.join(OUTPUT_DIR, f"NestedCV_{suffix}.xlsx")
    df.to_excel(out_file, index=False)
    print(f"Per-sample predictions saved to {out_file}")

    # Compute fold metrics
    fold_metrics = df.groupby(["repeat", "fold"]).apply(
        lambda g: pd.Series({
            "ACC": accuracy_score(g.y_true, g.y_pred),
            "BACC": balanced_accuracy_score(g.y_true, g.y_pred),
            "MCC": matthews_corrcoef(g.y_true, g.y_pred),
            "AUC": roc_auc_score(g.y_true, g.y_pred) if len(np.unique(g.y_true)) == 2 else np.nan
        })
    )

    print("\n===== FINAL NESTED CV RESULTS (50 folds) =====")
    print(f"ACC  = {fold_metrics['ACC'].mean():.4f} ± {fold_metrics['ACC'].std():.4f}")
    print(f"BACC = {fold_metrics['BACC'].mean():.4f} ± {fold_metrics['BACC'].std():.4f}")
    print(f"MCC  = {fold_metrics['MCC'].mean():.4f} ± {fold_metrics['MCC'].std():.4f}")
    print(f"AUC  = {fold_metrics['AUC'].mean():.4f} ± {fold_metrics['AUC'].std():.4f}")

    fold_out = os.path.join(OUTPUT_DIR, f"fold_metrics_{suffix}.xlsx")
    fold_metrics.to_excel(fold_out)
    print(f"Fold metrics saved to {fold_out}")

    return df, fold_metrics

# ============================================================================
# Running the 6 configurations
# ============================================================================

# ---- 5. Parallel + Attention ----
run_experiment('parallel', 'attention', 'parallel_attention')



<>:339: SyntaxWarning: invalid escape sequence '\s'
<>:340: SyntaxWarning: invalid escape sequence '\s'
<>:362: SyntaxWarning: invalid escape sequence '\s'
<>:363: SyntaxWarning: invalid escape sequence '\s'
<>:339: SyntaxWarning: invalid escape sequence '\s'
<>:340: SyntaxWarning: invalid escape sequence '\s'
<>:362: SyntaxWarning: invalid escape sequence '\s'
<>:363: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_2234/3525838335.py:339: SyntaxWarning: invalid escape sequence '\s'
  df1 = pd.read_csv(file_path1, sep='\s+', index_col=0)
/tmp/ipykernel_2234/3525838335.py:340: SyntaxWarning: invalid escape sequence '\s'
  df2 = pd.read_csv(file_path2, sep='\s+', index_col=0)
/tmp/ipykernel_2234/3525838335.py:362: SyntaxWarning: invalid escape sequence '\s'
  df1ca = pd.read_csv(file_path1ca, sep='\s+', header=None, index_col=False)
/tmp/ipykernel_2234/3525838335.py:363: SyntaxWarning: invalid escape sequence '\s'
  df2ca = pd.read_csv(file_path2ca, sep='\s+', header=None, ind

Loading data...
Data loaded. Number of subjects: 44
Using device: cuda

Running: arch=parallel, fusion=attention (suffix=parallel_attention)
OUTER SEED = 2772633

===== REPEAT 1 - FOLD 1 =====
Epoch 0: train loss = 0.6894, val_acc = 0.5556
Epoch 10: train loss = 0.6145, val_acc = 0.7778
Epoch 20: train loss = 0.3299, val_acc = 0.8889
Epoch 30: train loss = 0.0685, val_acc = 0.7778
Epoch 40: train loss = 0.0293, val_acc = 0.7778
Epoch 50: train loss = 0.0361, val_acc = 0.8889
Epoch 60: train loss = 0.0169, val_acc = 0.7778
Epoch 70: train loss = 0.0020, val_acc = 0.7778
Epoch 80: train loss = 0.0065, val_acc = 0.8889
Epoch 90: train loss = 0.0103, val_acc = 0.7778
Epoch 100: train loss = 0.0119, val_acc = 0.8889
Epoch 110: train loss = 0.0041, val_acc = 0.8889
Epoch 120: train loss = 0.0078, val_acc = 0.7778
Epoch 130: train loss = 0.0020, val_acc = 0.7778
Epoch 140: train loss = 0.0022, val_acc = 0.8889
Epoch 150: train loss = 0.0059, val_acc = 0.7778
Epoch 160: train loss = 0.0004, va

/tmp/ipykernel_2234/3525838335.py:502: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fold_metrics = df.groupby(["repeat", "fold"]).apply(


(     repeat  fold  outer_seed_global  inner_seed  sample_index  y_true  \
 0         1     1            2772633           1             3       0   
 1         1     1            2772633           1            12       0   
 2         1     1            2772633           1            14       0   
 3         1     1            2772633           1            17       0   
 4         1     1            2772633           1            22       1   
 ..      ...   ...                ...         ...           ...     ...   
 435      10     5            2772633          50            17       0   
 436      10     5            2772633          50            25       1   
 437      10     5            2772633          50            29       1   
 438      10     5            2772633          50            31       1   
 439      10     5            2772633          50            36       1   
 
      y_pred  train_cpu_energy_kWh  train_gpu_energy_kWh  train_ram_energy_kWh  \
 0         0    

In [ ]:
# ---- 6. Parallel + Sum ----
run_experiment('parallel', 'sum', 'parallel_sum')




Running: arch=parallel, fusion=sum (suffix=parallel_sum)
OUTER SEED = 8386238

===== REPEAT 1 - FOLD 1 =====
Epoch 0: train loss = 0.6784, val_acc = 0.5556
Epoch 10: train loss = 0.5034, val_acc = 0.8889
Epoch 20: train loss = 0.2576, val_acc = 0.8889
Epoch 30: train loss = 0.1562, val_acc = 0.8889
Epoch 40: train loss = 0.1011, val_acc = 0.8889
Epoch 50: train loss = 0.0282, val_acc = 0.8889
Epoch 60: train loss = 0.0092, val_acc = 0.8889
Epoch 70: train loss = 0.0108, val_acc = 0.8889
Epoch 80: train loss = 0.0024, val_acc = 0.8889
Epoch 90: train loss = 0.0017, val_acc = 0.8889
Epoch 100: train loss = 0.0036, val_acc = 0.8889
Epoch 110: train loss = 0.0062, val_acc = 0.8889
Epoch 120: train loss = 0.0023, val_acc = 0.8889
Epoch 130: train loss = 0.0028, val_acc = 0.8889
Epoch 140: train loss = 0.0014, val_acc = 0.8889
Epoch 150: train loss = 0.0014, val_acc = 0.8889
Epoch 160: train loss = 0.0001, val_acc = 0.8889
Epoch 170: train loss = 0.0014, val_acc = 0.8889
Epoch 180: train lo

/tmp/ipykernel_2234/3525838335.py:502: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fold_metrics = df.groupby(["repeat", "fold"]).apply(


(     repeat  fold  outer_seed_global  inner_seed  sample_index  y_true  \
 0         1     1            8386238           1             1       0   
 1         1     1            8386238           1             5       0   
 2         1     1            8386238           1            10       0   
 3         1     1            8386238           1            19       0   
 4         1     1            8386238           1            21       1   
 ..      ...   ...                ...         ...           ...     ...   
 435      10     5            8386238          50            11       0   
 436      10     5            8386238          50            29       1   
 437      10     5            8386238          50            31       1   
 438      10     5            8386238          50            42       1   
 439      10     5            8386238          50            43       1   
 
      y_pred  train_cpu_energy_kWh  train_gpu_energy_kWh  train_ram_energy_kWh  \
 0         1    

In [ ]:
# ---- 7. Parallel + Concat ----
run_experiment('parallel', 'concat', 'parallel_concat')



In [ ]:
# ---- 8. Shared + Attention ----
run_experiment('shared', 'attention', 'shared_attention')




Running: arch=shared, fusion=attention (suffix=shared_attention)
OUTER SEED = 6231200

===== REPEAT 1 - FOLD 1 =====
Epoch 0: train loss = 0.6934, val_acc = 0.5556
Epoch 10: train loss = 0.6832, val_acc = 0.5556
Epoch 20: train loss = 0.6296, val_acc = 0.5556
Epoch 30: train loss = 0.4777, val_acc = 0.8889
Epoch 40: train loss = 0.3846, val_acc = 0.8889
Epoch 50: train loss = 0.3316, val_acc = 0.8889
Epoch 60: train loss = 0.2314, val_acc = 0.7778
Epoch 70: train loss = 0.1926, val_acc = 0.7778
Epoch 80: train loss = 0.2408, val_acc = 0.8889
Epoch 90: train loss = 0.1580, val_acc = 0.8889
Epoch 100: train loss = 0.1802, val_acc = 0.8889
Epoch 110: train loss = 0.1315, val_acc = 0.8889
Epoch 120: train loss = 0.1423, val_acc = 0.8889
Epoch 130: train loss = 0.1647, val_acc = 0.8889
Epoch 140: train loss = 0.1286, val_acc = 0.8889
Epoch 150: train loss = 0.1433, val_acc = 0.8889
Epoch 160: train loss = 0.1108, val_acc = 0.8889
Epoch 170: train loss = 1.2113, val_acc = 0.6667
Epoch 180: 

/tmp/ipykernel_2234/3525838335.py:502: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fold_metrics = df.groupby(["repeat", "fold"]).apply(


(     repeat  fold  outer_seed_global  inner_seed  sample_index  y_true  \
 0         1     1            6231200           1             1       0   
 1         1     1            6231200           1             5       0   
 2         1     1            6231200           1             8       0   
 3         1     1            6231200           1            17       0   
 4         1     1            6231200           1            21       1   
 ..      ...   ...                ...         ...           ...     ...   
 435      10     5            6231200          50            19       0   
 436      10     5            6231200          50            23       1   
 437      10     5            6231200          50            31       1   
 438      10     5            6231200          50            33       1   
 439      10     5            6231200          50            41       1   
 
      y_pred  train_cpu_energy_kWh  train_gpu_energy_kWh  train_ram_energy_kWh  \
 0         1    

In [ ]:
# ---- 9. Shared + Sum ----
run_experiment('shared', 'sum', 'shared_sum')




Running: arch=shared, fusion=sum (suffix=shared_sum)
OUTER SEED = 5152777

===== REPEAT 1 - FOLD 1 =====
Epoch 0: train loss = 0.6966, val_acc = 0.5556
Epoch 10: train loss = 0.6348, val_acc = 0.6667
Epoch 20: train loss = 0.5463, val_acc = 0.8889
Epoch 30: train loss = 0.3855, val_acc = 0.7778
Epoch 40: train loss = 0.2257, val_acc = 0.8889
Epoch 50: train loss = 0.1043, val_acc = 0.8889
Epoch 60: train loss = 0.1050, val_acc = 0.8889
Epoch 70: train loss = 0.0401, val_acc = 0.7778
Epoch 80: train loss = 0.0552, val_acc = 0.7778
Epoch 90: train loss = 0.0107, val_acc = 0.7778
Epoch 100: train loss = 0.0070, val_acc = 0.7778
Epoch 110: train loss = 0.0090, val_acc = 0.7778
Epoch 120: train loss = 0.0125, val_acc = 0.7778
Epoch 130: train loss = 0.0027, val_acc = 0.7778
Epoch 140: train loss = 0.0027, val_acc = 0.7778
Epoch 150: train loss = 0.0037, val_acc = 0.7778
Epoch 160: train loss = 0.0033, val_acc = 0.7778
Epoch 170: train loss = 0.0011, val_acc = 0.7778
Epoch 180: train loss =

/tmp/ipykernel_2234/3525838335.py:502: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fold_metrics = df.groupby(["repeat", "fold"]).apply(


(     repeat  fold  outer_seed_global  inner_seed  sample_index  y_true  \
 0         1     1            5152777           1             2       0   
 1         1     1            5152777           1             4       0   
 2         1     1            5152777           1             8       0   
 3         1     1            5152777           1            18       0   
 4         1     1            5152777           1            24       1   
 ..      ...   ...                ...         ...           ...     ...   
 435      10     5            5152777          50            19       0   
 436      10     5            5152777          50            23       1   
 437      10     5            5152777          50            26       1   
 438      10     5            5152777          50            36       1   
 439      10     5            5152777          50            39       1   
 
      y_pred  train_cpu_energy_kWh  train_gpu_energy_kWh  train_ram_energy_kWh  \
 0         1    

In [ ]:
# ---- 10. Shared + Concat ----
run_experiment('shared', 'concat', 'shared_concat')

print("\nAll experiments finished. Results saved in", OUTPUT_DIR)


Running: arch=shared, fusion=concat (suffix=shared_concat)
OUTER SEED = 7977345

===== REPEAT 1 - FOLD 1 =====
Epoch 0: train loss = 0.6870, val_acc = 0.5556
Epoch 10: train loss = 0.5432, val_acc = 1.0000
Epoch 20: train loss = 0.4214, val_acc = 1.0000
Epoch 30: train loss = 0.3024, val_acc = 1.0000
Epoch 40: train loss = 0.2695, val_acc = 1.0000
Epoch 50: train loss = 0.1619, val_acc = 1.0000
Epoch 60: train loss = 0.0648, val_acc = 1.0000
Epoch 70: train loss = 0.0193, val_acc = 0.8889
Epoch 80: train loss = 0.0103, val_acc = 1.0000
Epoch 90: train loss = 0.0066, val_acc = 1.0000
Epoch 100: train loss = 0.0005, val_acc = 1.0000
Epoch 110: train loss = 0.0001, val_acc = 1.0000
Epoch 120: train loss = 0.0005, val_acc = 1.0000
Epoch 130: train loss = 0.0001, val_acc = 1.0000
Epoch 140: train loss = 0.0000, val_acc = 1.0000
Epoch 150: train loss = 0.0001, val_acc = 1.0000
Epoch 160: train loss = 0.0001, val_acc = 1.0000
Epoch 170: train loss = 0.0005, val_acc = 0.8889
Epoch 180: train 

/tmp/ipykernel_2234/3525838335.py:502: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  fold_metrics = df.groupby(["repeat", "fold"]).apply(
